# Extra 4 - Usar embeddings pre-treinados

No projeto final voce provavelmente vai treinar embeddings numa etapa e
usa-los em outra. Os dois pontos onde a sintaxe costuma travar:

1. **Alinhar** a matriz pre-treinada a ordem do vocabulario do modelo novo.
2. **Carregar** com `from_pretrained` e decidir entre **congelar** e
   **fine-tuning**.

Para ter uma matriz "pre-treinada" de verdade, o notebook treina rapido um
skip-gram num corpus pequeno (mesma ideia do modulo 1) e usa o resultado.

Tente resolver antes de olhar o `_solucoes`.

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)

corpus = [
    "the meeting is tomorrow morning",
    "send me the meeting notes please",
    "call me after the meeting",
    "lunch after the meeting tomorrow",
    "bring your notes to the meeting",
    "the project meeting moved to friday",
    "win a free cash prize now",
    "claim your free prize today",
    "you won a free cash award",
    "free entry to win a prize now",
    "urgent claim your free cash prize",
    "call now to claim your reward",
]

## 4.1 Treinar rapido um skip-gram (matriz "pre-treinada")

Reaproveite a receita do modulo 1, so que compacta:

1. `tokens`, `pre_word2idx` (indices a partir de 0, sem PAD), `idx2word`.
2. Pares skip-gram com `window = 2`.
3. Modelo: `nn.Embedding(V, 24)` + `nn.Linear(24, V)`; `forward` = `linear(embedding(x))`.
4. Treine 150 epocas, `CrossEntropyLoss`, `Adam(lr=0.01)`.
5. `pre_matrix = model.embedding.weight.detach().numpy()`  -> formato `(V, 24)`.

In [ ]:
# 4.1

## 4.2 Similaridade e analogia

1. `cos(a, b)` = similaridade de cosseno.
2. `mais_parecidas(palavra, topn=3)` usando `pre_matrix` e `pre_word2idx`.
3. `analogia(a, b, c)` = palavra mais proxima de `vec(b) - vec(a) + vec(c)`
   (excluindo `a`, `b`, `c`). Teste `mais_parecidas("free")` e
   `analogia("meeting", "notes", "free")` — com corpus minusculo o resultado
   e instavel, a ideia e so exercitar a conta.

In [ ]:
# 4.2

## 4.3 Alinhar a matriz a um vocabulario novo

O modelo novo tem o proprio vocabulario, em outra ordem e com palavras que
podem nao estar na matriz pre-treinada.

Dado o `novo_vocab` abaixo (com `<PAD>` e `<UNK>`), monte
`aligned` de formato `(len(novo_vocab), 24)`:

- linha `0` (`<PAD>`): zeros
- para cada palavra: se estiver em `pre_word2idx`, copie a linha correspondente
- se nao estiver (inclui `<UNK>`): vetor aleatorio pequeno
  (`np.random.normal(0, 0.1, 24)`)

Conte quantas palavras foram encontradas (hits) e quantas nao (misses).

In [ ]:
# 4.3

## 4.4 Carregar no `nn.Embedding` (congelado)

1. `emb_frozen = nn.Embedding.from_pretrained(torch.tensor(aligned),
   freeze=True, padding_idx=0)`.
2. Imprima `emb_frozen.weight.requires_grad` (deve ser `False`).
3. Confirme que `emb_frozen(torch.tensor([[2, 3, 4]]))` bate com
   `aligned[[2, 3, 4]]`.

In [ ]:
# 4.4

## 4.5 Congelado x fine-tuning: um passo de gradiente

Para ver a diferenca na pratica, com uma perda de brincadeira
(`saida.pow(2).mean()`):

1. `emb_frozen`: `weight.requires_grad` e `False`, entao chamar `.backward()`
   na saida nem funciona (o `RuntimeError` confirma que nada ali treina).
2. `emb_ft = nn.Embedding.from_pretrained(torch.tensor(aligned), freeze=False,
   padding_idx=0)`: depois de um `backward`, `emb_ft.weight.grad` existe.
   Guarde `w_antes`, faca um passo de SGD manual (`lr=0.1`) e mostre que
   `emb_ft.weight` mudou **so nas linhas usadas** (2, 3, 4) — a linha 0
   continua protegida pelo `padding_idx`.

In [ ]:
# 4.5

## 4.6 Salvar a matriz alinhada

`np.save("aligned_matrix.npy", aligned)` e o `novo_vocab` em
`aligned_vocab.json`. No projeto final e esse par (matriz + vocab na mesma
ordem) que voce carrega no modelo.

In [ ]:
# 4.6

## Resumo

- Matriz pre-treinada so serve se as **linhas seguirem a ordem do `word2idx`**
  do modelo que vai usar; palavras ausentes -> vetor aleatorio pequeno (ou `<UNK>`).
- `from_pretrained(..., freeze=True)`: treina so o resto do modelo (bom com
  pouco dado). `freeze=False`: ajusta os embeddings tambem (precisa de mais dado).
- `padding_idx=0` continua protegendo a linha de padding mesmo em fine-tuning.